In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_openml
import seaborn as sns
import plotly.express as px
from sklearn.decomposition import PCA
import os

os.makedirs("../processed", exist_ok=True)

In [ ]:
# Load dataset
X, y = fetch_openml(
    "mnist_784",
    version=1,
    return_X_y=True,
    as_frame=False
)
y = y.astype(int)

In [ ]:
# ---  Visualize random images ---
def plot_random_images(X, y, n=10, title="Random MNIST samples"):
    idx = np.random.choice(len(X), size=n, replace=False)
    plt.figure(figsize=(n * 1.2, 2))
    for i, j in enumerate(idx):
        plt.subplot(1, n, i + 1)
        plt.imshow(X[j].reshape(28, 28), cmap="gray")
        plt.title(int(y[j]))
        plt.axis("off")
    plt.suptitle(title)
    plt.show()
plot_random_images(X, y, n=10, title="Random samples")

In [ ]:
# Official MNIST split
X_train_full = X[:60000]
y_train_full = y[:60000]

X_test = X[60000:]
y_test = y[60000:]

# split validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.1,
    random_state=42,
    stratify=y_train_full
)

In [ ]:
# --- shapes ---
print("Shapes:")
print("X:", X.shape, "y:", y.shape)

print("\nFirst split:")
print("Full train:", X_train_full.shape)
print("Test:", X_test.shape)

print("\nValidation split:")
print("Validation train:", X_train.shape)
print("Validation test:", X_val.shape)

In [ ]:
# All labels distributions
classes, counts = np.unique(y, return_counts=True)
print("Label distribution (entire dataset):")
for c, count in zip(classes, counts):
    print(f"Class {c}: {count}")

In [ ]:
# before normalisation
print("Min/Max pixel values (before normalization):")
print(X.min(), X.max())

# After normalisation
X_train_norm = (X_train / 255.0).astype("float32")
X_val_norm   = (X_val / 255.0).astype("float32")
X_test_norm  = (X_test / 255.0).astype("float32")
print("\nAfter normalization:")
print("Train min/max:", X_train_norm.min(), X_train_norm.max())

In [ ]:
np.save("../processed/X_train_norm.npy", X_train_norm)
np.save("../processed/X_val_norm.npy", X_val_norm)
np.save("../processed/X_test_norm.npy", X_test_norm)

np.save("../processed/y_train.npy", y_train)
np.save("../processed/y_val.npy", y_val)
np.save("../processed/y_test.npy", y_test)

## Visualization

In [ ]:
ydf = pd.DataFrame({"class": y_train})
sns.countplot(x="class", data=df)
plt.title("Distribution of classes")
plt.show()

In [ ]:
# 2D PCA
X = df.drop(columns=['class'])  # class  is label or target
y = df['class']                 # labels

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_train_norm)

pc_df = pd.DataFrame(principal_components, columns=['PC1', 'PC2'])
pc_df['class'] = y.values

# plot
plt.figure(figsize=(8, 6))
for label in pc_df['class'].unique():
    subset = pc_df[pc_df['class'] == label]
    plt.scatter(subset['PC1'], subset['PC2'], label=label, alpha=0.6, s=5)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Algorithm')
plt.legend()
plt.show()

In [ ]:
#3D PCA on Train Dataset
pca = PCA(n_components=3)
principal_components = pca.fit_transform(X_train_norm)

pc_df = pd.DataFrame(principal_components, columns=['PC1', 'PC2', 'PC3'])
pc_df['class'] = y.values

# plot
fig = px.scatter_3d(
    pc_df,
    x='PC1',
    y='PC2',
    z='PC3',
    color='class',
    opacity=0.6,
    size_max=5,
    title='3D PCA Algorithm Plot'
)

fig.show()